# Week 3: Multi-Layer MLP Probe (The Evolution)

## The Problem with Single-Layer Probing

**Finding from Layer Sweep:**
- Layer 31 was "best" but results still weren't good
- This suggests: **Information is distributed across layers**

**Limitation of Single-Layer Probe:**
```
P(Code) = σ(W · h_layer_k + b)
```
- Assumes all info is in layer k
- Ignores rich information in other layers
- Linear decision boundary

## The Solution: Multi-Layer MLP Probe

**Key Innovation:**
```
P(Code) = MLP([h_layer_8, h_layer_16, h_layer_24, h_layer_31])
```

**Why this works:**
1. ✅ **Layer 8**: Syntax, structure, brackets
2. ✅ **Layer 16**: Abstract semantics, intent
3. ✅ **Layer 24**: Task-specific representations
4. ✅ **Layer 31**: Pre-output mode encoding

**Non-linear learning:**
- MLP can learn: "If layer_8 shows `{` AND layer_24 shows uncertainty → CODE"
- Linear probe cannot capture these interactions

**Research Precedent:**
- Dalvi et al. (2019): Multi-layer probing for syntax
- Hewitt & Manning (2019): Structural probes across layers

---

In [ ]:
# Cell 1: Install
!pip install -q transformers torch accelerate scipy scikit-learn pandas matplotlib seaborn

In [ ]:
# Cell 2: Imports
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from typing import List, Dict, Tuple
from itertools import combinations
from transformers import AutoTokenizer, AutoModelForCausalLM
from scipy.stats import entropy as scipy_entropy, ttest_ind
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score, f1_score, classification_report
from sklearn.model_selection import cross_val_score, train_test_split
from tqdm.notebook import tqdm
import pickle

np.random.seed(42)
torch.manual_seed(42)
print("✅ Imports")

In [ ]:
# Cell 3: Load Model
MODEL_NAME = "codellama/CodeLlama-7b-Instruct-hf"
print(f"Loading {MODEL_NAME}...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto",
    low_cpu_mem_usage=True
)
model.eval()
vocab_size = len(tokenizer)

print(f"✅ Model loaded")
print(f"Vocabulary size: {vocab_size:,}")
print(f"Total layers: {model.config.num_hidden_layers}")

## Training Data

In [ ]:
# Cell 4: Training prompts

CODE_TRAINING_PROMPTS = [
    # Python
    "def calculate_sum(a, b):",
    "import pandas as pd",
    "for i in range(10):",
    "class UserManager:",
    "if __name__ == '__main__':",
    "return {",
    "df = pd.read_csv(",
    "plt.plot(x, y)",
    "print(f'Value:",
    "try:\n    data =",
    "x = [i for i in",
    "with open('file.txt')",
    "lambda x: x",
    "async def fetch():",
    "@app.route('/api')",
    
    # JavaScript
    "const data = await",
    "function handleClick() {",
    "export default",
    "Promise.all(",
    "const [state, setState] =",
    "useEffect(() => {",
    "arr.map(item =>",
    
    # SQL
    "SELECT * FROM users",
    "INSERT INTO orders",
    "UPDATE table SET",
    "CREATE TABLE users (",
    "JOIN orders ON",
    
    # CLI/DevOps
    "git commit -m",
    "docker build -t",
    "npm run build",
    "pip install",
    "kubectl apply -f",
    
    # ML/Data Science
    "model.fit(X, y)",
    "df.groupby(",
    "np.array([",
    "torch.nn.Linear(",
]

LANGUAGE_TRAINING_PROMPTS = [
    "The quick brown fox",
    "Explain the theory of",
    "Once upon a time",
    "The weather today is",
    "I need help with",
    "To bake a cake, you",
    "The capital of France",
    "History teaches us that",
    "In conclusion,",
    "Please describe the",
    "The main difference is",
    "However, we can see",
    "It is important to",
    "According to the study,",
    "For example,",
    "This suggests that",
    "On the other hand,",
    "Writing a good essay",
    "The meaning of life",
    "How do I cook",
    "What is the best way",
    "Can you tell me",
    "I am feeling happy",
    "The movie was great",
    "She walked into the",
    "The purpose of this",
    "Let me explain why",
    "First of all,",
    "Another important point",
    "In other words,",
    "To summarize,",
    "The reason for this",
    "Most people believe",
    "It seems that",
    "We should consider",
    "Many experts agree",
]

y = np.array([1] * len(CODE_TRAINING_PROMPTS) + [0] * len(LANGUAGE_TRAINING_PROMPTS))

print(f"Training data:")
print(f"  Code prompts: {len(CODE_TRAINING_PROMPTS)}")
print(f"  Language prompts: {len(LANGUAGE_TRAINING_PROMPTS)}")
print(f"  Total: {len(y)}")

## Multi-Layer State Extraction

In [ ]:
# Cell 5: Multi-layer extraction

# Strategic layer selection:
# - Layer 8: Early-middle (syntax, structure)
# - Layer 16: Middle (abstract semantics)
# - Layer 24: Late-middle (task representations)
# - Layer 31: Final (pre-output mode)
SELECTED_LAYERS = [8, 16, 24, 31]

def get_multi_layer_state(prompt: str, layers: List[int]) -> np.ndarray:
    """
    Extract and concatenate hidden states from multiple layers.
    
    Returns:
        Concatenated vector of shape (len(layers) * hidden_dim,)
    """
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model(**inputs, output_hidden_states=True)
    
    # Concatenate selected layers
    # outputs.hidden_states[0] = embeddings
    # outputs.hidden_states[i+1] = layer i
    states = [
        outputs.hidden_states[layer_idx + 1][:, -1, :].cpu().numpy()[0]
        for layer_idx in layers
    ]
    
    return np.concatenate(states)

print(f"Selected layers: {SELECTED_LAYERS}")
print(f"\nLayer roles:")
print(f"  Layer {SELECTED_LAYERS[0]}: Syntax, structure")
print(f"  Layer {SELECTED_LAYERS[1]}: Abstract semantics")
print(f"  Layer {SELECTED_LAYERS[2]}: Task representations")
print(f"  Layer {SELECTED_LAYERS[3]}: Pre-output mode")

In [ ]:
# Cell 6: Extract training data

print("Extracting multi-layer hidden states...")
print("This will take ~1-2 minutes...")

X_code = []
for prompt in tqdm(CODE_TRAINING_PROMPTS, desc="Code"):
    state = get_multi_layer_state(prompt, SELECTED_LAYERS)
    X_code.append(state)

X_lang = []
for prompt in tqdm(LANGUAGE_TRAINING_PROMPTS, desc="Language"):
    state = get_multi_layer_state(prompt, SELECTED_LAYERS)
    X_lang.append(state)

X = np.array(X_code + X_lang)

print(f"\n✅ Multi-layer states extracted")
print(f"X shape: {X.shape}")
print(f"  Samples: {X.shape[0]}")
print(f"  Features: {X.shape[1]} ({len(SELECTED_LAYERS)} layers × {X.shape[1] // len(SELECTED_LAYERS)} hidden_dim)")

## Baseline: Linear Probe on Concatenated States

In [ ]:
# Cell 7: Linear baseline

print("Training LINEAR probe on multi-layer concatenation...")

linear_probe = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)

# Cross-validation
cv_scores_linear = cross_val_score(linear_probe, X, y, cv=5)

# Train on full data for later use
linear_probe.fit(X, y)

print(f"\n✅ Linear Probe (Baseline)")
print(f"CV Accuracy: {cv_scores_linear.mean():.1%} (+/- {cv_scores_linear.std():.1%})")
print(f"Training Accuracy: {linear_probe.score(X, y):.1%}")

## MLP Probe (Non-Linear)

In [ ]:
# Cell 8: MLP probe with regularization

print("Training MLP probe (non-linear)...")
print("\nMLP Architecture:")
print("  Input: {} features (concatenated layers)".format(X.shape[1]))
print("  Hidden Layer 1: 128 units + ReLU + Dropout(0.3)")
print("  Hidden Layer 2: 64 units + ReLU + Dropout(0.3)")
print("  Output: 2 classes (code/language)")

# MLP with regularization to prevent overfitting
mlp_probe = MLPClassifier(
    hidden_layer_sizes=(128, 64),  # Two hidden layers
    activation='relu',
    solver='adam',
    alpha=0.001,  # L2 regularization
    batch_size='auto',
    learning_rate='adaptive',
    learning_rate_init=0.001,
    max_iter=1000,
    random_state=42,
    early_stopping=True,  # Stop if validation score doesn't improve
    validation_fraction=0.2,
    n_iter_no_change=10,
    verbose=False
)

# Cross-validation
cv_scores_mlp = cross_val_score(mlp_probe, X, y, cv=5)

# Train on full data
mlp_probe.fit(X, y)

print(f"\n✅ MLP Probe")
print(f"CV Accuracy: {cv_scores_mlp.mean():.1%} (+/- {cv_scores_mlp.std():.1%})")
print(f"Training Accuracy: {mlp_probe.score(X, y):.1%}")
print(f"Training Iterations: {mlp_probe.n_iter_}")

In [ ]:
# Cell 9: Comparison

print("\n" + "="*80)
print("LINEAR vs MLP PROBE COMPARISON")
print("="*80)

print(f"\nLinear Probe (Baseline):")
print(f"  CV Accuracy: {cv_scores_linear.mean():.1%} (+/- {cv_scores_linear.std():.1%})")

print(f"\nMLP Probe (Non-Linear):")
print(f"  CV Accuracy: {cv_scores_mlp.mean():.1%} (+/- {cv_scores_mlp.std():.1%})")

improvement = (cv_scores_mlp.mean() - cv_scores_linear.mean()) * 100
print(f"\nImprovement: {improvement:+.1f} percentage points")

if cv_scores_mlp.mean() > cv_scores_linear.mean():
    print("✅ MLP learns non-linear patterns between layers!")
    best_probe = mlp_probe
    best_probe_name = "MLP"
else:
    print("⚠️  Linear probe is sufficient (information is linearly separable)")
    best_probe = linear_probe
    best_probe_name = "Linear"

## Layer Combination Ablation

In [ ]:
# Cell 10: Test different layer combinations

print("\n" + "="*80)
print("LAYER COMBINATION ABLATION")
print("="*80)
print("\nTesting which layer combinations work best...")

# Test all combinations of 2, 3, and 4 layers
layer_pool = [8, 16, 24, 31]
ablation_results = []

print("\nTesting combinations:")
for n in [2, 3, 4]:
    for combo in combinations(layer_pool, n):
        combo_list = list(combo)
        
        # Extract states for this combination
        X_combo = []
        for prompt in CODE_TRAINING_PROMPTS + LANGUAGE_TRAINING_PROMPTS:
            state = get_multi_layer_state(prompt, combo_list)
            X_combo.append(state)
        X_combo = np.array(X_combo)
        
        # Train MLP
        probe = MLPClassifier(
            hidden_layer_sizes=(64, 32),  # Smaller for fewer features
            max_iter=500,
            random_state=42,
            early_stopping=True,
            validation_fraction=0.2
        )
        
        cv_scores = cross_val_score(probe, X_combo, y, cv=5)
        
        ablation_results.append({
            'layers': combo_list,
            'n_layers': len(combo_list),
            'cv_mean': cv_scores.mean(),
            'cv_std': cv_scores.std(),
        })
        
        print(f"  Layers {combo_list}: {cv_scores.mean():.1%} (+/- {cv_scores.std():.1%})")

df_ablation = pd.DataFrame(ablation_results)
best_combo_idx = df_ablation['cv_mean'].idxmax()
best_combo = df_ablation.loc[best_combo_idx]

print(f"\n🏆 Best Combination: {best_combo['layers']}")
print(f"Accuracy: {best_combo['cv_mean']:.1%} (+/- {best_combo['cv_std']:.1%})")

## PCA Visualization

In [ ]:
# Cell 11: PCA on multi-layer concatenation

print("\nGenerating PCA visualization for multi-layer states...")

pca = PCA(n_components=2)
X_pca = pca.fit_transform(X)

X_code_pca = X_pca[:len(CODE_TRAINING_PROMPTS)]
X_lang_pca = X_pca[len(CODE_TRAINING_PROMPTS):]

plt.figure(figsize=(10, 7))
plt.scatter(X_code_pca[:, 0], X_code_pca[:, 1],
            c='blue', label='Code Mode', s=100, alpha=0.6, edgecolors='black')
plt.scatter(X_lang_pca[:, 0], X_lang_pca[:, 1],
            c='red', label='Language Mode', s=100, alpha=0.6, edgecolors='black')
plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%} variance)', fontsize=12)
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%} variance)', fontsize=12)
plt.title(f'Multi-Layer State Separation (Layers {SELECTED_LAYERS})\nMLP Accuracy: {cv_scores_mlp.mean():.1%}',
          fontsize=14)
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('week3_multi_layer_pca.png', dpi=150)
plt.show()

print(f"✅ PCA visualization saved")
print(f"Variance explained: {pca.explained_variance_ratio_[:2].sum():.1%}")

## Save Best Probe

In [ ]:
# Cell 12: Save best probe

# Save probe and configuration
probe_config = {
    'probe': best_probe,
    'probe_type': best_probe_name,
    'selected_layers': SELECTED_LAYERS,
    'cv_accuracy': cv_scores_mlp.mean() if best_probe_name == 'MLP' else cv_scores_linear.mean(),
}

with open('best_multi_layer_probe.pkl', 'wb') as f:
    pickle.dump(probe_config, f)

print(f"✅ Best probe saved: best_multi_layer_probe.pkl")
print(f"   Type: {best_probe_name}")
print(f"   Layers: {SELECTED_LAYERS}")
print(f"   Accuracy: {probe_config['cv_accuracy']:.1%}")

## Test Examples & Detailed Evaluation

In [ ]:
# Cell 13: Test on held-out examples

print("\n" + "="*80)
print("TESTING ON NEW EXAMPLES")
print("="*80)

TEST_CASES = [
    # Code examples (not in training)
    ("@app.get('/users')", "code"),
    ("DROP TABLE users", "code"),
    ("import numpy as np", "code"),
    ("const result = await fetch(", "code"),
    ("model = LinearRegression()", "code"),
    
    # Language examples (not in training)
    ("I will get the milk from", "language"),
    ("The table is made of wood", "language"),
    ("She walked to the store", "language"),
    ("The best approach for learning", "language"),
    ("We need to understand", "language"),
]

print("\nTest Results:")
print("-" * 80)

correct = 0
for prompt, true_label in TEST_CASES:
    state = get_multi_layer_state(prompt, SELECTED_LAYERS)
    pred = best_probe.predict([state])[0]
    pred_label = "code" if pred == 1 else "language"
    
    # Get confidence
    if hasattr(best_probe, 'predict_proba'):
        proba = best_probe.predict_proba([state])[0]
        confidence = proba[pred]
    else:
        # For linear, use decision function
        score = best_probe.decision_function([state])[0]
        confidence = 1 / (1 + np.exp(-score))  # Sigmoid
    
    status = "✅" if pred_label == true_label else "❌"
    if pred_label == true_label:
        correct += 1
    
    print(f"{status} '{prompt[:40]:<40}' → {pred_label:8s} (conf: {confidence:.1%}, true: {true_label})")

print("-" * 80)
print(f"Test Accuracy: {correct}/{len(TEST_CASES)} = {correct/len(TEST_CASES):.1%}")

## Summary & Recommendations

In [ ]:
# Cell 14: Final summary

print("\n" + "="*80)
print("MULTI-LAYER MLP PROBE: FINAL SUMMARY")
print("="*80)

print("\n1. APPROACH")
print(f"   - Concatenated layers: {SELECTED_LAYERS}")
print(f"   - Feature dimension: {X.shape[1]}")
print(f"   - Training samples: {len(y)}")

print("\n2. RESULTS")
print(f"   - Linear probe: {cv_scores_linear.mean():.1%}")
print(f"   - MLP probe: {cv_scores_mlp.mean():.1%}")
print(f"   - Best: {best_probe_name}")

print("\n3. LAYER ABLATION")
print(f"   - Best combination: {best_combo['layers']}")
print(f"   - Best accuracy: {best_combo['cv_mean']:.1%}")

print("\n4. INTERPRETATION")
if cv_scores_mlp.mean() > 0.85:
    print("   ✅ EXCELLENT: Multi-layer probe works well!")
    print("   → Information IS distributed across layers")
    print("   → Non-linear interactions are important")
elif cv_scores_mlp.mean() > 0.70:
    print("   ⚠️  MODERATE: Multi-layer helps but not dramatically")
    print("   → Some information is distributed")
    print("   → May need better training data or different layers")
else:
    print("   ❌ POOR: Multi-layer probe still struggling")
    print("   → Problem may not be solvable by probing alone")
    print("   → Consider alternative approaches")

print("\n5. NEXT STEPS")
if cv_scores_mlp.mean() > 0.80:
    print("   → Integrate with CCE spike detection")
    print("   → Run full Week 3 experiment")
    print("   → Use saved probe: best_multi_layer_probe.pkl")
else:
    print("   → Investigate why probe is failing")
    print("   → Try different layer combinations")
    print("   → Consider more training data")
    print("   → May need different approach")

print("\n" + "="*80)